# Nowcasting Canadian Consumption with Card Spending Data

This notebook walks through a nowcasting exercise for **quarterly Canadian consumption growth** using monthly retail sales and VISA card spending data.

The question is whether high-frequency payments data — specifically the VISA Spending Momentum Index (SMI) — adds value beyond what cumulative retail sales growth already provides.

We proceed in three steps:

1. **Explore the data** — understand the structure and visualise the key variables.
2. **Single-day example** — walk through the forecasting procedure for one month.
3. **Full evaluation** — run the expanding-window exercise and assess performance by month of quarter.

## 1. Data exploration

The dataset has one row per **month** from 2017 to 2025. Each row contains:

| Variable | Description |
|---|---|
| `date` | First day of the month |
| `quarter` | Start date of the quarter this month belongs to |
| `moq` | Month of quarter (1, 2, or 3) |
| `consumption_growth` | Realised quarterly consumption growth, q/q % (same for every month in a quarter) |
| `retail_cum_growth` | Cumulative retail sales growth within the quarter up to this month |
| `visa_headline` | VISA Spending Momentum Index (SMI) headline value |

**Data sources.** `retail_cum_growth` is derived from total retail sales published by Statistics Canada (CANSIM Table 20-10-0008-01, vector V1446870183). The `retail_cum_growth` construction computes the cumulative mean within the quarter relative to last quarter's average. This is the **traditional monthly indicator**.

`visa_headline` is the VISA Spending Momentum Index (SMI), a seasonally adjusted index (100 = baseline) that Visa publishes monthly based on its proprietary transaction data. It measures spending *momentum* rather than levels, capturing acceleration and deceleration in Canadian consumer spending. This is the **alternative high-frequency indicator** from the private sector.

Note the data is **monthly** (not daily, unlike the Euro area sentiment exercise). This means each quarter contributes exactly 3 observations, and training sets are matched by month-of-quarter rather than day-of-quarter.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

df = pd.read_csv("canada_nowcast_data.csv", parse_dates=["date", "quarter"])

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Quarters: {df['quarter'].nunique()}")
df.head(10)

### Time series of the key variables

To get a quarterly view, we take the **end-of-quarter** values (month 3) for retail sales growth and the VISA index, and plot them against realised consumption growth.

In [ ]:
# End-of-quarter snapshots
qly = df[df["moq"] == 3].set_index("quarter")[["consumption_growth", "retail_cum_growth", "visa_headline"]].copy()

# Normalize to zero mean, unit std for comparison
growth_z = (qly["consumption_growth"] - qly["consumption_growth"].mean()) / qly["consumption_growth"].std()
retail_z = (qly["retail_cum_growth"] - qly["retail_cum_growth"].mean()) / qly["retail_cum_growth"].std()
visa_z = (qly["visa_headline"] - qly["visa_headline"].mean()) / qly["visa_headline"].std()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Consumption growth and retail sales (normalized)
ax = axes[0]
ax.plot(qly.index, growth_z, "k-", label="Consumption growth (z)")
ax.plot(qly.index, retail_z, "C0--", label="Retail sales cum. growth (z)", alpha=0.7)
ax.axhline(0, color="grey", linewidth=0.5)
ax.legend()
ax.set_ylabel("Std. deviations")
ax.set_title("Consumption growth vs retail sales")

# Consumption growth and VISA SMI (normalized)
ax = axes[1]
ax.plot(qly.index, growth_z, "k-", label="Consumption growth (z)")
ax.plot(qly.index, visa_z, "C1--", label="VISA SMI headline (z)", alpha=0.7)
ax.axhline(0, color="grey", linewidth=0.5)
ax.legend()
ax.set_ylabel("Std. deviations")
ax.set_title("Consumption growth vs VISA SMI")

plt.tight_layout()
plt.show()

## 2. Model definitions

We nowcast quarterly consumption growth $y_q$ using information available in month $m$ of quarter $q$.

### Benchmark model (retail sales only)

$$\hat{y}_{q,m}^{\text{bench}} = \hat{\theta}_0 + \hat{\theta}_1 \, R_{q,m}$$

where $R_{q,m}$ is the cumulative retail sales growth through month $m$ of quarter $q$.

### Alternative model (retail sales + VISA)

$$\hat{y}_{q,m}^{\text{alt}} = \hat{\theta}_0 + \hat{\theta}_1 \, R_{q,m} + \hat{\theta}_2 \, V_{q,m}$$

where $V_{q,m}$ is the VISA SMI headline value in month $m$.

### Training data construction

For each forecast month, we train on **one observation per past quarter** matched by month-of-quarter (`moq`). For example, if we are forecasting in month 2 of 2023-Q3, we train on all past quarters' month-2 observations. This ensures training data reflects a comparable information set.

Both models are estimated by OLS on an expanding window of past quarters.

## 3. Single-month example

Before running the full evaluation, let's pick a single forecast month and walk through the procedure step by step.

In [ ]:
# Pick a specific month
forecast_date = pd.Timestamp("2022-05-01")
row = df.loc[df["date"] == forecast_date].iloc[0]

print(f"Forecast date:         {row['date'].date()}")
print(f"Quarter:               {row['quarter'].date()}")
print(f"Month of quarter:      {int(row['moq'])}")
print(f"Realised growth:       {row['consumption_growth']:.3f}")
print(f"Retail cum. growth:    {row['retail_cum_growth']:.3f}")
print(f"VISA SMI:              {row['visa_headline']:.3f}")

### Construct the training set

We select all past quarters' observations that share the same month-of-quarter position.

In [ ]:
target = "consumption_growth"
bench_vars = ["retail_cum_growth"]
alt_vars = ["retail_cum_growth", "visa_headline"]

quart = row["quarter"]
moq_today = int(row["moq"])

# Keep only months from past quarters with the same moq
past = df[(df["quarter"] < quart) & (df["moq"] == moq_today)].copy()
train = past[[target] + alt_vars].dropna()

print(f"Training observations: {len(train)} (one per past quarter, moq={moq_today})")
print(f"Date range: {past['date'].min().date()} to {past['date'].max().date()}")
print()
train.describe()

### Fit both models and compare

In [ ]:
# Benchmark: retail only
X_bench_train = train[bench_vars].values
y_train = train[target].values

model_bench = LinearRegression().fit(X_bench_train, y_train)
pred_bench = model_bench.predict(row[bench_vars].values.reshape(1, -1))[0]

# Alternative: retail + VISA
X_alt_train = train[alt_vars].values

model_alt = LinearRegression().fit(X_alt_train, y_train)
pred_alt = model_alt.predict(row[alt_vars].values.reshape(1, -1))[0]

actual = row[target]

print(f"Actual consumption growth:  {actual:.3f}")
print(f"Benchmark prediction:       {pred_bench:.3f}  (error: {actual - pred_bench:+.3f})")
print(f"Alternative prediction:     {pred_alt:.3f}  (error: {actual - pred_alt:+.3f})")

## 4. Full expanding-window evaluation

Now we run the same procedure for every month from 2021-Q1 onward, producing a monthly nowcast from both models.

In [ ]:
start_quarter = pd.Timestamp("2021-01-01")
forecast_rows = df.index[df["quarter"] >= start_quarter]

target = "consumption_growth"
bench_vars = ["retail_cum_growth"]
alt_vars = ["retail_cum_growth", "visa_headline"]

results = []

for tt in forecast_rows:
    row = df.iloc[tt]
    quart = row["quarter"]
    moq_today = row["moq"]

    # Skip if predictors missing
    if row[alt_vars].isna().any():
        continue

    # Training: one row per past quarter, matched by moq
    past = df[(df["quarter"] < quart) & (df["moq"] == moq_today)].copy()
    train = past[[target] + alt_vars].dropna()

    if len(train) < 4:
        continue

    X_bench = train[bench_vars].values
    X_alt = train[alt_vars].values
    y = train[target].values

    # Benchmark: retail only
    model_bench = LinearRegression().fit(X_bench, y)
    pred_bench = model_bench.predict(row[bench_vars].values.reshape(1, -1))[0]

    # Alternative: retail + VISA
    model_alt = LinearRegression().fit(X_alt, y)
    pred_alt = model_alt.predict(row[alt_vars].values.reshape(1, -1))[0]

    results.append({
        "date": row["date"],
        "quarter": quart,
        "moq": moq_today,
        "consumption_growth": row[target],
        "pred_bench": pred_bench,
        "pred_alt": pred_alt,
    })

preds = pd.DataFrame(results)
print(f"Evaluation from: {start_quarter.date()}")
print(f"Predictions: {len(preds)} months")

### Overall results

In [ ]:
preds["se_bench"] = (preds["consumption_growth"] - preds["pred_bench"]) ** 2
preds["se_alt"] = (preds["consumption_growth"] - preds["pred_alt"]) ** 2

mse_bench = preds["se_bench"].mean()
mse_alt = preds["se_alt"].mean()
gain = (1 - mse_alt / mse_bench) * 100

print("Overall: Retail benchmark vs Retail + VISA")
print("=" * 50)
print(f"MSE (retail only):     {mse_bench:.4f}")
print(f"MSE (retail + VISA):   {mse_alt:.4f}")
print(f"Gain:                  {gain:.2f}%")
print()
print("Positive gain = VISA helps beyond retail sales alone.")

### Results by month of quarter

As with the Euro area exercise, the key question is whether the alternative data source is most valuable **early in the quarter**, when less traditional data is available.

In [ ]:
by_moq = preds.groupby("moq").agg(
    mse_bench=("se_bench", "mean"),
    mse_alt=("se_alt", "mean"),
    n=("se_bench", "count"),
).reset_index()
by_moq["gain_pct"] = ((1 - by_moq["mse_alt"] / by_moq["mse_bench"]) * 100).round(2)

print("By month of quarter")
print("=" * 55)
print(by_moq.to_string(index=False))
print()
print("Month 1 = early in quarter, Month 3 = late.")

### MSE by month of quarter

We can visualise the gain from VISA across the three months of the quarter.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# MSE comparison
ax = axes[0]
x = by_moq["moq"]
width = 0.35
ax.bar(x - width/2, by_moq["mse_bench"], width, label="Retail only", color="C3", alpha=0.7)
ax.bar(x + width/2, by_moq["mse_alt"], width, label="Retail + VISA", color="C2", alpha=0.7)
ax.set_xlabel("Month of quarter")
ax.set_ylabel("MSE")
ax.set_xticks([1, 2, 3])
ax.legend()
ax.set_title("MSE by month of quarter")

# Gain
ax = axes[1]
ax.bar(by_moq["moq"], by_moq["gain_pct"], color="C0", alpha=0.7)
ax.axhline(0, color="grey", linewidth=0.5)
ax.set_xlabel("Month of quarter")
ax.set_ylabel("Gain (%)")
ax.set_xticks([1, 2, 3])
ax.set_title("MSE gain from adding VISA (%)")

plt.tight_layout()
plt.show()

### Relationship between VISA SMI and consumption growth

To understand the VISA signal, we plot end-of-quarter VISA values against realised consumption growth.

In [ ]:
# End-of-quarter snapshots
qly = df[df["moq"] == 3].copy()

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(qly["visa_headline"], qly["consumption_growth"],
           color="grey", alpha=0.6, edgecolors="k", linewidths=0.5)

# OLS fit
from numpy.polynomial.polynomial import polyfit
b, m = polyfit(qly["visa_headline"], qly["consumption_growth"], 1)
x_line = np.linspace(qly["visa_headline"].min(), qly["visa_headline"].max(), 100)
ax.plot(x_line, b + m * x_line, "b--", label="OLS fit")

ax.set_xlabel("VISA SMI headline (end of quarter)")
ax.set_ylabel("Consumption growth (q/q %)")
ax.set_title("VISA SMI vs consumption growth")
ax.legend()
plt.tight_layout()
plt.show()

### Predictions vs actuals over time

Finally, we plot the time series of predictions from both models against realised consumption growth.

In [ ]:
# Use end-of-quarter predictions (moq == 3) for a clean quarterly view
qly_preds = preds[preds["moq"] == 3].copy()

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(qly_preds["quarter"], qly_preds["consumption_growth"], "k-o",
        label="Actual", markersize=4)
ax.plot(qly_preds["quarter"], qly_preds["pred_bench"], "C3--s",
        label="Retail only", markersize=3, alpha=0.7)
ax.plot(qly_preds["quarter"], qly_preds["pred_alt"], "C2--^",
        label="Retail + VISA", markersize=3, alpha=0.7)

ax.axhline(0, color="grey", linewidth=0.5)
ax.set_xlabel("Quarter")
ax.set_ylabel("Consumption growth (q/q %)")
ax.set_title("Nowcast predictions vs actuals (end-of-quarter)")
ax.legend()
plt.tight_layout()
plt.show()